# 1. USTC: Spanish-language editions, 1500–1700

I download every Spanish-language edition in the Universal Short Title Catalogue, subject by subject, from the public search
pages of ustc.ac.uk (20 editions per page). The USTC lists no Spanish editions after 1700. An edition filed under two subjects
comes twice; notebook 3 keeps one copy. If the download stops, running the notebook again resumes it.

---

# 1. USTC: ediciones en castellano, 1500–1700

Descargo todas las ediciones en castellano del Universal Short Title Catalogue, materia por materia, desde las páginas públicas
de búsqueda de ustc.ac.uk (20 ediciones por página). El USTC no registra ediciones en castellano después de 1700. Una edición
clasificada en dos materias aparece dos veces; en el notebook 3 me quedo con una sola. Si la descarga se corta, vuelvo a correr
el notebook y retoma donde quedó.

In [1]:
# Celda 1
# Paths and parameters:
%cd "E:/No sincroniza/Software desarrollado/spanish-imprints-counts"
DATA = "E:/Datos/spanish_imprints"   # Raw downloads go here, outside the repo:
LANGUAGE = "Spanish"
YEAR_FROM, YEAR_TO = 1500, 1700       # The USTC has no Spanish editions after 1700.
OUT = DATA + "/ustc_spanish.jsonl"    # One edition per line, tagged with its USTC subject.
PAUSE = 1                             # Seconds between requests.

E:\No sincroniza\Software desarrollado\spanish-imprints-counts


In [2]:
# Celda 2
import subprocess, re, html, json, time, os, urllib.parse
from collections import Counter

# Reads one results page (20 editions) of the USTC "explore" search:
def ustc_page(subject, pg):
    params = dict(q="", fql=LANGUAGE, fqyf=YEAR_FROM, fqyt=YEAR_TO, pg=pg)
    if subject:
        params["fqs"] = subject
    url = "https://www.ustc.ac.uk/explore?" + urllib.parse.urlencode(params)
    for attempt in range(40):  # The USTC sometimes goes down for a while (500 Proxy Error): wait up to ~40 min.
        text = subprocess.run(["curl", "-s", "-m", "60", "-A", "Mozilla/5.0", url], capture_output=True).stdout.decode("utf8", "replace")
        m = re.search(r'data-page="([^"]*)"', text)
        if m:
            return json.loads(html.unescape(m.group(1)))["props"]["results"]
        time.sleep(60)
    raise RuntimeError(url)

In [3]:
# Celda 3
# Subjects and their counts, from the facets of the unfiltered search:
first = ustc_page(None, 1)
total = first["meta"]["NumFound"]
subjects = {f["label"]: f["options"]["value"] for f in first["facets"]["subject"]}
print("Editions:", total, "| sum over subjects:", sum(subjects.values()))
for s, n in subjects.items():
    print(f"{n:>7}  {s}")

Editions: 84345 | sum over subjects: 97832
  29081  Religious
  23192  Jurisprudence
   9678  News Books
   6459  Ordinances and Edicts
   3932  Poetry
   3467  Sermons
   3369  History and Chronicles
   3075  Literature
   1704  Drama
   1399  Multilingual Book
   1089  Petitions
   1029  Newspapers
   1028  Philosophy and Morality
   1013  Medical Texts
    739  Funeral Orations
    702  Educational Books
    679  Economics
    617  Commemorative Events
    593  Dictionaries
    529  Government and Political Thought
    436  Classical Authors
    366  Heraldic Works and Genealogies
    347  Bibles (including parts)
    320  Travel
    310  Military Handbooks
    305  Adages and Emblem Books
    275  Music
    266  Science
    258  Astrology and Cosmography
    254  Almanacs, Calendars and Prognostications
    203  Political Tracts
    186  Agriculture
    174  Linguistics and Philology
    137  Mathematics
    108  Art and Architecture
     84  Women, Gender and Marriage
     67  Boo

In [4]:
# Celda 4
# Downloads every subject page by page; reruns resume where they stopped:
done = set()
if os.path.exists(OUT):
    for line in open(OUT, encoding="utf8"):
        r = json.loads(line)
        done.add((r["_subject"], r["_pg"]))

os.makedirs(DATA, exist_ok=True)
with open(OUT, "a", encoding="utf8") as f:
    for subject, n in subjects.items():
        last_page = (n + 19) // 20
        for pg in range(1, last_page + 1):
            if (subject, pg) in done:
                continue
            results = ustc_page(subject, pg)
            for x in results["data"]:
                f.write(json.dumps(dict(id=x["id"], **x["attributes"], _subject=subject, _pg=pg), ensure_ascii=False) + "\n")
            f.flush()
            time.sleep(PAUSE)
        print("done:", subject, n, flush=True)

done: Religious 29081
done: Jurisprudence 23192
done: News Books 9678
done: Ordinances and Edicts 6459
done: Poetry 3932
done: Sermons 3467
done: History and Chronicles 3369
done: Literature 3075
done: Drama 1704
done: Multilingual Book 1399
done: Petitions 1089
done: Newspapers 1029
done: Philosophy and Morality 1028
done: Medical Texts 1013
done: Funeral Orations 739
done: Educational Books 702
done: Economics 679
done: Commemorative Events 617
done: Dictionaries 593
done: Government and Political Thought 529
done: Classical Authors 436
done: Heraldic Works and Genealogies 366
done: Bibles (including parts) 347
done: Travel 320
done: Military Handbooks 310
done: Adages and Emblem Books 305
done: Music 275
done: Science 266
done: Astrology and Cosmography 258
done: Almanacs, Calendars and Prognostications 254
done: Political Tracts 203
done: Agriculture 186
done: Linguistics and Philology 174
done: Mathematics 137
done: Art and Architecture 108
done: Women, Gender and Marriage 84
done

In [5]:
# Celda 5
# Check: editions downloaded per subject against the facet counts:
got = Counter()
ids = set()
for line in open(OUT, encoding="utf8"):
    r = json.loads(line)
    got[r["_subject"]] += 1
    ids.add(r["id"])
for s, n in subjects.items():
    if got[s] != n:
        print("mismatch:", s, "facet", n, "downloaded", got[s])
print("Lines:", sum(got.values()), "| unique editions:", len(ids), "| USTC total:", total)

Lines: 97832 | unique editions: 82768 | USTC total: 84345
